# 01 | Fontes acadêmicas e mapeamento

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** importar ou inspecionar referências. **Entradas:** três planilhas em `/reference`, montadas somente para leitura. Nenhuma macro é executada. **Esperado:** 107 origens; 31 entidades/175 campos no arquivo real.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from pathlib import Path
from src.academic.importer import import_sources,persist
from src.persistence.db import rows
IMPORTAR=True
if IMPORTAR:
    imported=import_sources(Path('/reference'))
    persist(imported)
display(pd.DataFrame(rows('SELECT filename,sha256,byte_count FROM planning.academic_source_files ORDER BY filename')))


,filename,sha256,byte_count
0,DADOS-EXTERNOS-OTIMIZACAO-HORARIOS-V3.xlsx,96f100462e90da37f1bd90ebc65a41ad16697fc05076cd...,99453
1,Planilha de Turmas 2024.2_CiberSegurança.xlsm,cd5fb5f08956c55cf33ce2e28b877a45059f56c059f4c8...,718276
2,Planilha de Turmas 2024.2_Engenharia de Comput...,8c472d8ecf2468197a3cb980bf6d1e18f1297d43b651e2...,727194


**Executar:** ler o que foi efetivamente persistido. **Resultado:** cursos distintos, fontes e divergências do contrato.

In [2]:

from src.workbench.academic_ui import references
source=pd.DataFrame(references())
assert len(source)==107
display(source.groupby(['course_code','shift']).size().rename('linhas').reset_index())
display(source[['course_code','curriculum_code','discipline_code','activity_type','division_label','source_row_number','status']].head(12))
display(rows('SELECT count(DISTINCT entity_name) AS entidades,count(*) AS campos FROM planning.academic_data_dictionary'))


,course_code,shift,linhas
0,2248,Manhã,42
1,2287,Noite,25
2,2348,Noite,40


,course_code,curriculum_code,discipline_code,activity_type,division_label,source_row_number,status
0,2248,CECP 2023/2,P185,Prática,P2-1,6,REFERENCE_ONLY
1,2248,CECP 2023/2,P185,Prática,P2-2,7,REFERENCE_ONLY
2,2248,CECP 2024/1,P185,Prática,P2-1,8,REFERENCE_ONLY
3,2248,CECP 2024/1,P185,Prática,P2-2,9,REFERENCE_ONLY
4,2248,CECP 2023/1,P164,Teórica,Completa,10,REFERENCE_ONLY
5,2248,CECP 2023/1,P164,Prática,P2-1,11,REFERENCE_ONLY
6,2248,CECP 2023/1,P164,Prática,P2-2,12,REFERENCE_ONLY
7,2248,CECP 2023/1,PCO149,Prática,P2-1,13,REFERENCE_ONLY
8,2248,CECP 2023/1,PCO149,Prática,P2-2,14,REFERENCE_ONLY
9,2248,CECP 2022/2,PCO125A,Prática,P1-1,15,REFERENCE_ONLY


[{'entidades': 31, 'campos': 175}]

**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.